In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_absolute_error,root_mean_squared_error
import pickle

In [2]:
df = pd.read_csv('cleaned_swiggy.csv')

In [3]:
df.rename(columns={'Time_taken(min)':'time_taken'},inplace=True)

In [4]:
df.drop(columns=['ID', 'Delivery_person_ID','Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 'Order_Date' ,'Year','Day','City_name','Month'],inplace=True)

In [5]:
df['distance_type']=df.Distance.round().apply(lambda x: 'short' if x in range(0,5) else 'medium' if x in range(5,10) else 'long' if x in range(10,15) else 'very long' if x in range(15,25) else np.nan)

In [6]:
df.dropna(inplace=True)

In [7]:
df.columns

Index(['Delivery_person_Age', 'Delivery_person_Ratings', 'Weatherconditions',
       'Road_traffic_density', 'Vehicle_condition', 'Type_of_order',
       'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City',
       'time_taken', 'pickup_time', 'order_time_of_day', 'is_weekend',
       'Distance', 'distance_type'],
      dtype='object')

In [8]:
Xd = df.drop(columns='time_taken')
yd= df['time_taken']

In [9]:
X_traind, X_testd, y_traind, y_testd = train_test_split(Xd, yd, test_size=0.2, random_state=42)

In [10]:
num_cols =['Delivery_person_Age', 'Delivery_person_Ratings','pickup_time','Distance']
nominal_cat_cols = ['Weatherconditions', 'Type_of_order',
       'Type_of_vehicle', 'Festival', 'City',  'order_time_of_day', 
       'is_weekend']
ordinal_cat_cols = ['distance_type','Road_traffic_density']

In [11]:
# define order for ordinal variables
distance_type_order = ['short','medium','long','very long']
road_traffic_order = ['low','medium','high','jam']

In [12]:
# build a preprocessor
preprocessor_not_impute = ColumnTransformer(transformers=[
    ("scale", MinMaxScaler(), num_cols),
    ("nominal_encode", OneHotEncoder(drop="first",handle_unknown="ignore",sparse_output=False), nominal_cat_cols),
    ("ordinal_encode", OrdinalEncoder(categories=[distance_type_order,road_traffic_order,]), ordinal_cat_cols)
],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)

preprocessor_not_impute.set_output(transform="pandas")

ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                  remainder='passthrough',
                  transformers=[('scale', MinMaxScaler(),
                                 ['Delivery_person_Age',
                                  'Delivery_person_Ratings', 'pickup_time',
                                  'Distance']),
                                ('nominal_encode',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Weatherconditions', 'Type_of_order',
                                  'Type_of_vehicle', 'Festival', 'City',
                                  'order_time_of_day', 'is_weekend']),
                                ('ordinal_encode',
                                 OrdinalEncoder(categories=[['short', 'medium',
                                                             'long',
                                                             'very long'],
                                                            ['low', 'medium',
                                                             'high', 'jam']]),
                                 ['distance_type', 'Road_traffic_density'])],
                  verbose_feature_names_out=False)

In [13]:
# transform the data

X_train_transd = preprocessor_not_impute.fit_transform(X_traind)
X_test_transd = preprocessor_not_impute.transform(X_testd)

In [92]:
X_train_transd.columns

Index(['Delivery_person_Age', 'Delivery_person_Ratings', 'pickup_time',
       'Distance', 'Weatherconditions_Fog', 'Weatherconditions_Sandstorms',
       'Weatherconditions_Stormy', 'Weatherconditions_Sunny',
       'Weatherconditions_Windy', 'Type_of_order_drinks', 'Type_of_order_meal',
       'Type_of_order_snack', 'Type_of_vehicle_motorcycle',
       'Type_of_vehicle_scooter', 'Festival_Yes ', 'City_semi-urban',
       'City_urban', 'order_time_of_day_Afternoon',
       'order_time_of_day_Evening', 'order_time_of_day_Morning',
       'order_time_of_day_Night', 'is_weekend_1', 'distance_type',
       'Road_traffic_density', 'Vehicle_condition', 'multiple_deliveries'],
      dtype='object')

In [14]:
# transform target column

pt = PowerTransformer()

y_train_ptd = pt.fit_transform(y_traind.values.reshape(-1,1))
y_test_ptd = pt.transform(y_testd.values.reshape(-1,1))

In [15]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.compose import TransformedTargetRegressor

In [16]:
best_xgb_params = {'n_estimators': 327,
 'max_depth': 24,
 'min_samples_split': 2,
 'learning_rate': 0.31714572861651974,
 'max_leaves': 30}

best_lgbm_params = {'n_estimators': 151,
 'max_depth': 35,
 'learning_rate': 0.14176038430301802,
 'subsample': 0.5063794781773334,
 'min_child_weight': 16,
 'min_split_gain': 0.032638264298539235,
 'reg_lambda': 77.2128094212829}
best_xgb = XGBRegressor(**best_xgb_params)
best_lgbm = LGBMRegressor(**best_lgbm_params)

In [17]:
# build transformed regressor
voting_reg = VotingRegressor(estimators=[("xgb",best_xgb),("lgbm",best_lgbm)],n_jobs=-1)
model2 = TransformedTargetRegressor(regressor=voting_reg,
                                    transformer=pt)
model2.fit(X_train_transd,y_traind)

TransformedTargetRegressor(regressor=VotingRegressor(estimators=[('xgb',
                                                                  XGBRegressor(base_score=None,
                                                                               booster=None,
                                                                               callbacks=None,
                                                                               colsample_bylevel=None,
                                                                               colsample_bynode=None,
                                                                               colsample_bytree=None,
                                                                               device=None,
                                                                               early_stopping_rounds=None,
                                                                               enable_categorical=False,
                                                                               eval_metric=None,
                                                                               feature_types=None,
                                                                               gamma=None,
                                                                               grow_policy=None,
                                                                               importance_type=None,
                                                                               inter...
                                                                               missing=nan,
                                                                               monotone_constraints=None,
                                                                               multi_strategy=None,
                                                                               n_estimators=327,
                                                                               n_jobs=None,
                                                                               num_parallel_tree=None, ...)),
                                                                 ('lgbm',
                                                                  LGBMRegressor(learning_rate=0.14176038430301802,
                                                                                max_depth=35,
                                                                                min_child_weight=16,
                                                                                min_split_gain=0.032638264298539235,
                                                                                n_estimators=151,
                                                                                reg_lambda=77.2128094212829,
                                                                                subsample=0.5063794781773334))],
                                                     n_jobs=-1),
                           transformer=PowerTransformer())

In [18]:
# get the train and test predictions

y_train_pred = model2.predict(X_train_transd)
y_test_pred = model2.predict(X_test_transd)

# calculate the train and test mae

train_mae = mean_absolute_error(y_traind,y_train_pred)
test_mae = mean_absolute_error(y_testd,y_test_pred)

# calculate the r2 scores

train_r2 = r2_score(y_traind,y_train_pred)
test_r2 = r2_score(y_testd,y_test_pred)

print(f"Train and test mae are {train_mae} and {test_mae}, also train and test r2 are {train_r2} and {test_r2}")

Train and test mae are 2.7172338901921 and 3.1067544933420113, also train and test r2 are 0.8679599162405811 and 0.8296141409354394


In [21]:
model_pipe = Pipeline(steps=[
                                ("preprocessing",preprocessor_not_impute ),
                                ("model",model2)
                            ])

model_pipe

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                                   remainder='passthrough',
                                   transformers=[('scale', MinMaxScaler(),
                                                  ['Delivery_person_Age',
                                                   'Delivery_person_Ratings',
                                                   'pickup_time', 'Distance']),
                                                 ('nominal_encode',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['Weatherconditions',
                                                   'Type_of_...
                                                                                                monotone_constraints=None,
                                                                                                multi_strategy=None,
                                                                                                n_estimators=327,
                                                                                                n_jobs=None,
                                                                                                num_parallel_tree=None, ...)),
                                                                                  ('lgbm',
                                                                                   LGBMRegressor(learning_rate=0.14176038430301802,
                                                                                                 max_depth=35,
                                                                                                 min_child_weight=16,
                                                                                                 min_split_gain=0.032638264298539235,
                                                                                                 n_estimators=151,
                                                                                                 reg_lambda=77.2128094212829,
                                                                                                 subsample=0.5063794781773334))],
                                                                      n_jobs=-1),
                                            transformer=PowerTransformer()))])

In [55]:
model_pipe.fit(X_traind,y_traind)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                                   remainder='passthrough',
                                   transformers=[('scale', MinMaxScaler(),
                                                  ['Delivery_person_Age',
                                                   'Delivery_person_Ratings',
                                                   'pickup_time', 'Distance']),
                                                 ('nominal_encode',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['Weatherconditions',
                                                   'Type_of_...
                                                                                                monotone_constraints=None,
                                                                                                multi_strategy=None,
                                                                                                n_estimators=327,
                                                                                                n_jobs=None,
                                                                                                num_parallel_tree=None, ...)),
                                                                                  ('lgbm',
                                                                                   LGBMRegressor(learning_rate=0.14176038430301802,
                                                                                                 max_depth=35,
                                                                                                 min_child_weight=16,
                                                                                                 min_split_gain=0.032638264298539235,
                                                                                                 n_estimators=151,
                                                                                                 reg_lambda=77.2128094212829,
                                                                                                 subsample=0.5063794781773334))],
                                                                      n_jobs=-1),
                                            transformer=PowerTransformer()))])

In [56]:
y_train_pred2= model_pipe.predict(X_traind)
y_test_pred2 = model_pipe.predict(X_testd)

In [57]:
train_mae = mean_absolute_error(y_traind,y_train_pred2)
test_mae = mean_absolute_error(y_testd,y_test_pred2)

# calculate the r2 scores

train_r2 = r2_score(y_traind,y_train_pred2)
test_r2 = r2_score(y_testd,y_test_pred2)

print(f"Train and test mae are {train_mae} and {test_mae}, also train and test r2 are {train_r2} and {test_r2}")

Train and test mae are 2.7172338901921 and 3.1067544933420113, also train and test r2 are 0.8679599162405811 and 0.8296141409354394


In [31]:
X_traind.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30451 entries, 24939 to 18803
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Delivery_person_Age      30451 non-null  float64
 1   Delivery_person_Ratings  30451 non-null  float64
 2   Weatherconditions        30451 non-null  object 
 3   Road_traffic_density     30451 non-null  object 
 4   Vehicle_condition        30451 non-null  int64  
 5   Type_of_order            30451 non-null  object 
 6   Type_of_vehicle          30451 non-null  object 
 7   multiple_deliveries      30451 non-null  float64
 8   Festival                 30451 non-null  object 
 9   City                     30451 non-null  object 
 10  pickup_time              30451 non-null  int64  
 11  order_time_of_day        30451 non-null  object 
 12  is_weekend               30451 non-null  int64  
 13  Distance                 30451 non-null  float64
 14  distance_type          

In [44]:
X_traind.distance_type

24939       medium
9439        medium
18409       medium
15267        short
43924        short
           ...    
20042         long
7521     very long
13483        short
1038     very long
18803         long
Name: distance_type, Length: 30451, dtype: object

In [61]:
filename='swiggy_model_pipe2.sav'
pickle.dump(model_pipe,open(filename,'wb'))

In [12]:
loaded_model = pickle.load(open('swiggy_model_pipe2.sav','rb'))

In [13]:
X_traind.City.unique()

array(['urban', 'metropolitian', 'semi-urban'], dtype=object)

In [14]:
# Your input data
input_data = [22, 4.2, 'Stormy', 'medium', 2, 'buffet', 'motorcycle', 1.0, 'Yes', 'metropolitian', '10', 'Morning', 0, 12.3, 'medium']

# Column names expected by your model (example — replace with actual column names)
column_names =['Delivery_person_Age', 'Delivery_person_Ratings', 'Weatherconditions',
       'Road_traffic_density', 'Vehicle_condition', 'Type_of_order',
       'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City',
       'pickup_time', 'order_time_of_day', 'is_weekend', 'Distance',
       'distance_type']

# Create a DataFrame
input_df = pd.DataFrame([input_data], columns=column_names)

# Make prediction
prediction = loaded_model.predict(input_df)
print(prediction)


[29.86238644]


In [15]:
type(loaded_model)

sklearn.pipeline.Pipeline

In [16]:
X_traind.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30451 entries, 24939 to 18803
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Delivery_person_Age      30451 non-null  float64
 1   Delivery_person_Ratings  30451 non-null  float64
 2   Weatherconditions        30451 non-null  object 
 3   Road_traffic_density     30451 non-null  object 
 4   Vehicle_condition        30451 non-null  int64  
 5   Type_of_order            30451 non-null  object 
 6   Type_of_vehicle          30451 non-null  object 
 7   multiple_deliveries      30451 non-null  float64
 8   Festival                 30451 non-null  object 
 9   City                     30451 non-null  object 
 10  pickup_time              30451 non-null  int64  
 11  order_time_of_day        30451 non-null  object 
 12  is_weekend               30451 non-null  int64  
 13  Distance                 30451 non-null  float64
 14  distance_type          

In [23]:
df.distance_type.unique()

array(['short', 'very long', 'medium', 'long'], dtype=object)